# Install

In [1]:
# 1. Kiểm tra xem Kaggle đã nhận GPU chưa
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

PyTorch version: 2.11.0+cu128
CUDA Available: True
Device name: Tesla T4


In [2]:
# 2. Cài đặt PyTorch Geometric & các thư viện bổ trợ
!pip install torch_geometric
!pip install nilearn scikit-learn numpy pandas matplotlib seaborn

# 3. Cài các gói mở rộng của PyG tự động tương thích với bản CUDA GPU hiện tại
!pip install torch_scatter torch_sparse torch_cluster -f https://data.pyg.org/whl/torch-{torch.__version__}.html

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 87.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 91.0 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 98.0 MB/s eta 0:00:00:00:01


# 1. Data pipeline: FC -> Graph

In [3]:
## Load FC Matrix
import numpy as np

dataset_path = '/kaggle/input/datasets/kietkolat/fc-matric-for-graph/abide_cc200_fc_matrices.npz'

data = np.load(dataset_path)
fc_matrices = data['fc_matrices']
y = data['y']

print(f"✅ Loaded fc_matrices shape: {fc_matrices.shape}") # (n_subjects, 200, 200)
print(f"✅ Loaded y shape: {y.shape}")                     # (n_subjects,)

✅ Loaded fc_matrices shape: (871, 200, 200)
✅ Loaded y shape: (871,)


In [22]:
import numpy as np
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader


def proportional_threshold(fc_matrix, density=0.2):
    """Return boolean mask of edges to keep (undirected, no self-loops)."""
    n = fc_matrix.shape[0]
    triu_idx = np.triu_indices(n, k=1)
    vals = fc_matrix[triu_idx]
    abs_vals = np.abs(vals)
    thresh = np.percentile(abs_vals, 100 * (1 - density))
    mask = np.abs(fc_matrix) >= thresh
    np.fill_diagonal(mask, False)
    return mask


def fc_to_graph(fc_matrix, label, threshold=0.3, use_proportional=True, density=0.2):
    """
    fc_matrix: (n_rois, n_rois) symmetric FC (Pearson r or Fisher-z)
    label:     0 or 1
    """
    n_rois = fc_matrix.shape[0]

    # 1. Node features: row-wise stats
    node_mean = fc_matrix.mean(axis=1)
    node_std  = fc_matrix.std(axis=1)
    x = np.stack([node_mean, node_std], axis=1)  # (n_rois, 2)
    x = torch.tensor(x, dtype=torch.float)

    # 2. Edge construction
    if use_proportional:
        mask = proportional_threshold(fc_matrix, density=density)
    else:
        mask = np.abs(fc_matrix) > threshold
        np.fill_diagonal(mask, False)

    rows, cols = np.where(mask)

    # Make undirected: keep upper triangle then add reverse
    upper = rows < cols
    rows_u, cols_u = rows[upper], cols[upper]
    edge_index_u = torch.tensor(np.array([rows_u, cols_u]), dtype=torch.long)
    edge_attr_u  = torch.tensor(fc_matrix[rows_u, cols_u], dtype=torch.float)

    # Duplicate reverse edges
    edge_index = torch.cat([edge_index_u, edge_index_u.flip(0)], dim=1)
    edge_attr  = torch.cat([edge_attr_u, edge_attr_u], dim=0)

    # 3. Label
    y = torch.tensor([label], dtype=torch.long)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)


def build_graph_dataset(fc_matrices, labels, threshold=0.3, use_proportional=True, density=0.2):
    """
    fc_matrices: (n_subjects, n_rois, n_rois)
    labels:      (n_subjects,)
    """
    graph_list = []
    for i in range(len(labels)):
        g = fc_to_graph(
            fc_matrices[i],
            labels[i],
            threshold=threshold,
            use_proportional=use_proportional,
            density=density
        )
        graph_list.append(g)
    return graph_list

In [23]:
# graph_dataset = build_graph_dataset(fc_matrices, y, threshold=0.52)

graph_dataset = build_graph_dataset(fc_matrices, y, use_proportional=True, density=0.2)

g = graph_dataset[0]
print(f"Nodes: {g.num_nodes}")
print(f"Edges: {g.num_edges}")
print(f"Node features: {g.x.shape}")
print(f"Label: {g.y.item()}")
print(f"Edge density: {g.num_edges / (200*199):.1%}")

Nodes: 200
Edges: 7960
Node features: torch.Size([200, 2])
Label: 1
Edge density: 20.0%


# 2. Analyze Graph properties

In [24]:
import matplotlib.pyplot as plt


thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
avg_edges  = []


for thr in thresholds:
    edges_list = []
    for fc in fc_matrices:
        n_edges = np.sum(np.abs(fc) > thr) - 200  # trừ self-loops
        edges_list.append(n_edges)
    avg_edges.append(np.mean(edges_list))
    print(f"Threshold={thr}: avg edges={np.mean(edges_list):.0f} "
          f"({np.mean(edges_list)/(200*199):.1%} density)")

Threshold=0.1: avg edges=31806 (79.9% density)
Threshold=0.2: avg edges=23881 (60.0% density)
Threshold=0.3: avg edges=16422 (41.3% density)
Threshold=0.4: avg edges=10086 (25.3% density)
Threshold=0.5: avg edges=5361 (13.5% density)


# 3. GCN

In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool

class BrainGCN(nn.Module):
    """
    GCN cho brain connectivity classification.
    Kiến trúc: 3 lớp GCN → Global Pooling → MLP
    """
    def __init__(self, in_channels=2, hidden_dim=64, num_classes=2, dropout=0.5):
        super(BrainGCN, self).__init__()
        
        # ── GCN Layers ──
        self.conv1 = GCNConv(in_channels, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim // 2)
        
        # ── Batch Normalization ──
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim // 2)
        
        # ── Classifier ──
        # Concat mean + max pooling → 2 * (hidden_dim // 2)
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.fc2 = nn.Linear(32, num_classes)
        
        self.dropout = nn.Dropout(p=dropout)
    
    def forward(self, x, edge_index, batch):
        # ── Graph Convolution ──
        h = self.conv1(x, edge_index)
        h = self.bn1(h)
        h = F.relu(h)
        h = self.dropout(h)
        
        h = self.conv2(h, edge_index)
        h = self.bn2(h)
        h = F.relu(h)
        h = self.dropout(h)
        
        h = self.conv3(h, edge_index)
        h = self.bn3(h)
        h = F.relu(h)
        
        # ── Readout: Mean + Max Pooling (concat) ──
        h_mean = global_mean_pool(h, batch)   # (batch_size, hidden//2)
        h_max  = global_max_pool(h, batch)    # (batch_size, hidden//2)
        h_graph = torch.cat([h_mean, h_max], dim=1)  # (batch_size, hidden)
        
        # ── MLP Classification ──
        h_graph = self.dropout(h_graph)
        h_graph = F.relu(self.fc1(h_graph))
        h_graph = self.dropout(h_graph)
        out = self.fc2(h_graph)
        
        return out

# 4. GNN

In [31]:
from torch_geometric.nn import GATConv

class BrainGAT(nn.Module):
    """
    GAT với multi-head attention — học ROI nào quan trọng hơn.
    Advantage: Attention weights có thể visualize → giải thích được.
    """
    def __init__(self, in_channels=2, hidden_dim=32, 
                 num_heads=4, num_classes=2, dropout=0.5):
        super(BrainGAT, self).__init__()
        
        # ── GAT Layers ──
        # Layer 1: concat multi-head → hidden_dim * num_heads output
        self.gat1 = GATConv(in_channels, hidden_dim, heads=num_heads,
                             dropout=dropout, concat=True)
        # Layer 2: average multi-head → hidden_dim output  
        self.gat2 = GATConv(hidden_dim * num_heads, hidden_dim, heads=1,
                             dropout=dropout, concat=False)
        
        self.bn1 = nn.BatchNorm1d(hidden_dim * num_heads)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        
        # ── Classifier ──
        self.fc1 = nn.Linear(hidden_dim * 2, 32)
        self.fc2 = nn.Linear(32, num_classes)
        
        self.dropout = nn.Dropout(p=dropout)
    
    def forward(self, x, edge_index, batch):
        # ── GAT Layers ──
        h = self.gat1(x, edge_index)   # (n_nodes, hidden*heads)
        h = self.bn1(h)
        h = F.elu(h)
        h = self.dropout(h)
        
        h = self.gat2(h, edge_index)   # (n_nodes, hidden)
        h = self.bn2(h)
        h = F.elu(h)
        
        # ── Readout ──
        h_mean = global_mean_pool(h, batch)
        h_max  = global_max_pool(h, batch)
        h_graph = torch.cat([h_mean, h_max], dim=1)
        
        # ── Classifier ──
        h_graph = self.dropout(h_graph)
        h_graph = F.elu(self.fc1(h_graph))
        h_graph = self.dropout(h_graph)
        out = self.fc2(h_graph)
        
        return out
    
    def get_attention_weights(self, x, edge_index):
        """Trả về attention weights để visualize kết nối quan trọng."""
        _, (edge_index_out, alpha) = self.gat1(
            x, edge_index, return_attention_weights=True
        )
        return edge_index_out, alpha

# 5. Training loop

In [32]:
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, f1_score
import numpy as np

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    
    for batch in loader:
        batch = batch.to(device)
        out  = model(batch.x, batch.edge_index, batch.batch)
        prob = F.softmax(out, dim=1)[:, 1].cpu().numpy()
        pred = out.argmax(dim=1).cpu().numpy()
        
        y_true.extend(batch.y.cpu().numpy())
        y_pred.extend(pred)
        y_prob.extend(prob)
    
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    auc     = roc_auc_score(y_true, y_prob)
    f1      = f1_score(y_true, y_pred)
    
    return bal_acc, auc, f1


def run_cv_gnn(graph_dataset, y, n_splits=5, 
               model_class=BrainGCN, epochs=100,
               lr=1e-3, weight_decay=1e-4,
               batch_size=32, device='cpu'):
    """
    5-Fold Stratified CV cho GNN.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    fold_results = {
        'val_bal_acc': [], 'val_auc': [], 'val_f1': [],
        'train_bal_acc': [], 'train_auc': [], 'train_f1': []
    }
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(y)), y)):
        print(f"\n{'─'*50}")
        print(f"  Fold {fold+1}/{n_splits}")
        print(f"{'─'*50}")
        
        # ── Data Split ──
        train_data = [graph_dataset[i] for i in train_idx]
        val_data   = [graph_dataset[i] for i in val_idx]
        
        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
        val_loader   = DataLoader(val_data,   batch_size=batch_size, shuffle=False)
        
        # ── Model & Optimizer ──
        model     = model_class().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)
        
        # Class weights để xử lý imbalance
        y_train = y[train_idx]
        pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()]).to(device)
        criterion = nn.CrossEntropyLoss()
        
        # ── Training Loop ──
        best_val_auc = 0
        best_state   = None
        
        for epoch in range(1, epochs + 1):
            loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
            scheduler.step()
            
            if epoch % 20 == 0:
                val_bal_acc, val_auc, val_f1 = evaluate(model, val_loader, device)
                print(f"  Epoch {epoch:3d} | Loss: {loss:.4f} | "
                      f"Val BalAcc: {val_bal_acc:.3f} | AUC: {val_auc:.3f}")
                
                if val_auc > best_val_auc:
                    best_val_auc = val_auc
                    best_state   = {k: v.clone() for k, v in model.state_dict().items()}
        
        # ── Evaluate Best Model ──
        model.load_state_dict(best_state)
        train_metrics = evaluate(model, train_loader, device)
        val_metrics   = evaluate(model, val_loader,   device)
        
        fold_results['train_bal_acc'].append(train_metrics[0])
        fold_results['train_auc'].append(train_metrics[1])
        fold_results['train_f1'].append(train_metrics[2])
        fold_results['val_bal_acc'].append(val_metrics[0])
        fold_results['val_auc'].append(val_metrics[1])
        fold_results['val_f1'].append(val_metrics[2])
        
        print(f"\n  ✅ Best → Val BalAcc: {val_metrics[0]:.3f} | AUC: {val_metrics[1]:.3f}")
    
    return fold_results

# 6. Experiments

In [33]:
# ── Config ──
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
THRESHOLD = 0.3
EPOCHS    = 100

print(f"Device: {DEVICE}")
print(f"Graph threshold: {THRESHOLD}")

# ── Tạo dataset ──
graph_dataset = build_graph_dataset(fc_matrices, y, threshold=THRESHOLD)

# ── Exp 3a: GCN ──
print("\n" + "="*55)
print("   EXP 3a — BrainGCN")
print("="*55)
results_gcn = run_cv_gnn(
    graph_dataset, y,
    model_class=BrainGCN,
    epochs=EPOCHS,
    device=DEVICE
)

# ── Exp 3b: GAT ──
print("\n" + "="*55)
print("   EXP 3b — BrainGAT")
print("="*55)
results_gat = run_cv_gnn(
    graph_dataset, y,
    model_class=BrainGAT,
    epochs=EPOCHS,
    device=DEVICE
)

# ── Tổng hợp kết quả ──
print("\n" + "="*65)
print("   TỔNG KẾT QUẢ — GNN vs SVM Baseline")
print("="*65)

comparison = {
    'SVM Baseline':  {'BalAcc': 64.5, 'AUC': 70.0, 'F1': 60.7},
    'GCN (Exp 3a)':  {
        'BalAcc': np.mean(results_gcn['val_bal_acc']) * 100,
        'AUC':    np.mean(results_gcn['val_auc']) * 100,
        'F1':     np.mean(results_gcn['val_f1']) * 100
    },
    'GAT (Exp 3b)':  {
        'BalAcc': np.mean(results_gat['val_bal_acc']) * 100,
        'AUC':    np.mean(results_gat['val_auc']) * 100,
        'F1':     np.mean(results_gat['val_f1']) * 100
    },
}

for model_name, metrics in comparison.items():
    print(f"\n  {model_name}:")
    for metric, val in metrics.items():
        print(f"    {metric:<12}: {val:.1f}%")

Device: cuda
Graph threshold: 0.3

   EXP 3a — BrainGCN

──────────────────────────────────────────────────
  Fold 1/5
──────────────────────────────────────────────────
  Epoch  20 | Loss: 0.6898 | Val BalAcc: 0.500 | AUC: 0.419
  Epoch  40 | Loss: 0.6868 | Val BalAcc: 0.500 | AUC: 0.431
  Epoch  60 | Loss: 0.6901 | Val BalAcc: 0.492 | AUC: 0.458
  Epoch  80 | Loss: 0.6868 | Val BalAcc: 0.494 | AUC: 0.453
  Epoch 100 | Loss: 0.6888 | Val BalAcc: 0.483 | AUC: 0.462

  ✅ Best → Val BalAcc: 0.483 | AUC: 0.462

──────────────────────────────────────────────────
  Fold 2/5
──────────────────────────────────────────────────
  Epoch  20 | Loss: 0.6934 | Val BalAcc: 0.514 | AUC: 0.444
  Epoch  40 | Loss: 0.6927 | Val BalAcc: 0.495 | AUC: 0.453
  Epoch  60 | Loss: 0.6884 | Val BalAcc: 0.485 | AUC: 0.468
  Epoch  80 | Loss: 0.6930 | Val BalAcc: 0.477 | AUC: 0.461
  Epoch 100 | Loss: 0.6870 | Val BalAcc: 0.489 | AUC: 0.464

  ✅ Best → Val BalAcc: 0.485 | AUC: 0.468

─────────────────────────────

# 7. Effect of threshold

In [ ]:
thresholds_to_test = [0.1, 0.2, 0.3, 0.4, 0.5]
ablation_results   = {}

for thr in thresholds_to_test:
    print(f"\n--- Threshold = {thr} ---")
    dataset_thr = build_graph_dataset(fc_matrices, y, threshold=thr)
    res = run_cv_gnn(dataset_thr, y, model_class=BrainGAT,
                     epochs=80, device=DEVICE)
    ablation_results[thr] = np.mean(res['val_bal_acc']) * 100
    print(f"  BalAcc = {ablation_results[thr]:.1f}%")

# Plot
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.plot(list(ablation_results.keys()), 
         list(ablation_results.values()), 'bo-', linewidth=2, markersize=8)
plt.axhline(64.5, color='red', linestyle='--', label='SVM Baseline (64.5%)')
plt.xlabel('FC Threshold', fontsize=12)
plt.ylabel('Balanced Accuracy (%)', fontsize=12)
plt.title('Ablation: Graph Threshold vs Performance')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('exp3_ablation_threshold.png', dpi=120)
plt.show()

# 8. Visualization: Attention Weights

In [37]:
from nilearn import plotting
import numpy as np

def visualize_attention(model, graph_dataset, idx=0, top_k=20):
    """
    Hiển thị top-k kết nối não quan trọng nhất theo GAT attention.
    """
    model.eval()
    g = graph_dataset[idx].to('cpu')
    
    with torch.no_grad():
        edge_idx, alpha = model.get_attention_weights(g.x, g.edge_index)
    
    # Lấy trung bình attention qua các heads
    alpha_mean = alpha.mean(dim=1).numpy()
    
    # Top-k edges
    top_k_idx = np.argsort(alpha_mean)[-top_k:]
    src_nodes  = edge_idx[0, top_k_idx].numpy()
    dst_nodes  = edge_idx[1, top_k_idx].numpy()
    weights    = alpha_mean[top_k_idx]
    
    # Tạo connectivity matrix từ attention weights
    attention_matrix = np.zeros((200, 200))
    for s, d, w in zip(src_nodes, dst_nodes, weights):
        attention_matrix[s, d] = w
        attention_matrix[d, s] = w
    
    # Plot
    plt.figure(figsize=(10, 8))
    plt.imshow(attention_matrix, cmap='hot', aspect='auto')
    plt.colorbar(label='Attention Weight')
    plt.title(f'Top-{top_k} Connections theo GAT Attention\n'
              f'(Subject: {"ASD" if g.y.item() == 1 else "TD"})')
    plt.xlabel('ROI Index')
    plt.ylabel('ROI Index')
    plt.tight_layout()
    plt.savefig('exp3_attention_map.png', dpi=120)
    plt.show()

# Visualize cho 1 subject ASD và 1 subject TD
asd_idx = np.where(y == 1)[0][0]
td_idx  = np.where(y == 0)[0][0]
visualize_attention(model_gat, graph_dataset, idx=asd_idx, top_k=20)

NameError: name 'model_gat' is not defined